# Cloud Profiling Radar (CPR_CLD_2A) — Antarctica / Ross Sea

Produces **`CPR_CLD_2A_1s_AN.nc`**: 1-second-resampled `land_flag`,
`ice_water_path` and `liquid_water_path` over the Ross Sea, ready for
`merger_AN.ipynb`.

**Region:** 80-60 deg S, 160 deg E - 140 deg W (= 160-220 in the 0-360 convention),
polar orbit **frame `G`**. The antimeridian is handled by filtering in the
0-360 convention after the search.

Adapted from `cpr_cld_2a_EP.ipynb`. Requires a MAAP bearer token in `token.txt`.

In [ ]:
from pystac_client import Client
import fsspec
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import requests
from IPython.display import Image, display
import os
import pathlib

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

In [ ]:
EC_COLLECTION = ['EarthCAREL2Validated_MAAP']

In [ ]:
# Region of interest — Ross Sea (Antarctica). See markdown above.
LAT_MIN, LAT_MAX = -80, -60
LON360_MIN, LON360_MAX = 160, 220   # 160 deg E ... 140 deg W in the 0-360 convention
FRAME = 'G'                          # EarthCARE polar orbit frame

## Search the catalog

Polar frame `G`, no bbox (antimeridian). Inspect `search.matched()` before
removing `max_items`.

In [ ]:
search = catalog.search(
    collections=EC_COLLECTION,
    filter=f"productType = 'CPR_CLD_2A' and frame = '{FRAME}'",
    method='GET',
    max_items=1000,
)

items = list(search.items())
print(f"Accessing {len(items)} items (limited by max_items).")
print(f"{search.matched()} items found that matched the query.")

In [ ]:
data = search.item_collection_as_dict()

df = pd.json_normalize(data, record_path=['features'])[
    [
        "id",
        "properties.product:type",
        "properties.updated",
        "assets.product.href",
        "assets.quicklook.href",
        "assets.enclosure_1.href",
        "assets.enclosure_2.href",
    ]
]

df.rename(columns={
    'properties.product:type': 'product_type',
    'properties.updated': 'last_modified',
    'assets.product.href': 'Zipped Product',
    'assets.quicklook.href': 'quicklook_url',
    'assets.enclosure_1.href': 'h5_url',
    'assets.enclosure_2.href': 'HDR_url',
}, inplace=True)

df.sort_values(by='id', ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)
df

In [ ]:
with open("token.txt", "rt") as f:
    token = f.read().strip().replace("\n", "")

In [ ]:
nooffiles = len(df)
# nooffiles = 10  # uncomment for a quick test run

## Download and concatenate

In [ ]:
frames = []
for fileno in tqdm(range(nooffiles)):
    ds_url = df.loc[fileno, "h5_url"]

    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
    with fs.open(ds_url, "rb") as f:
        ds = xr.open_dataset(f, engine="h5netcdf", group="ScienceData").compute()

        selection = ds[['latitude', 'longitude', 'land_flag',
                        'ice_water_path', 'liquid_water_path']]
        selection = selection.assign_coords(along_track=ds['time'])
        selection = selection.rename({"along_track": "time"})

        frames.append(selection)

## Restrict to the Ross Sea

Frame `G` granules cover the whole polar band, so we keep only the
observations inside the Ross Sea box. Longitudes are converted to the
0-360 convention so the antimeridian crossing is a single contiguous
interval (160-220). The very same filter is re-applied in
`kmeans_AN.ipynb`, so this stays consistent end-to-end.

In [ ]:
radar = xr.concat(frames, dim='time').sortby('time')

lon360 = radar.longitude % 360
radar = radar.where(
    (radar.latitude > LAT_MIN) & (radar.latitude < LAT_MAX) &
    (lon360 > LON360_MIN) & (lon360 < LON360_MAX),
    drop=True,
)

radar_1s = radar.resample(time='1s').mean().dropna(dim='time', how='all')
radar_1s

In [ ]:
radar_1s.time.plot(marker='o')

## Save

In [ ]:
radar_1s.to_netcdf('CPR_CLD_2A_1s_AN.nc')

In [ ]:
%cp "CPR_CLD_2A_1s_AN.nc" "/home/jovyan/my-private-bucket/."